# Drainage Network Optimization — Colab demo

Small-budget demonstration of the SC mini project. It runs the **whole method**
end-to-end inside Colab's limits: encode the Bellinge network as binary genes,
evolve conduit subsets with **NSGA-II**, judge each layout with a real **SWMM**
simulation, and plot the trade-off front.

It never touches the huge time-series zips — the SWMM `.inp` model is enough, and
the search runs on a fast **sub-catchment** with a tiny budget. Heavy/full runs are
done locally (see the project README).

## 1. Install dependencies

In [ ]:
!pip install -q pymoo pyswmm swmmio pandas pyarrow matplotlib scipy requests tqdm pyyaml

## 2. Get the project code

Either clone the repo (edit the URL to yours) **or** upload the
`drainage-network-optimization/` folder via the Colab file browser.

In [ ]:
# Option A — clone (edit to your repo), then cd into the project folder:
# !git clone https://github.com/<you>/<repo>.git
# %cd <repo>/drainage-network-optimization

# Option B — if you uploaded the folder:
%cd drainage-network-optimization
!ls src

## 3. Download just the SWMM model (small — skips the big zips)

In [ ]:
!python -m src.download_data --inp-only

## 4. Tiny demo budget + build the sub-catchment graph

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['scope'] = 'sub'
cfg['subcatchment']['max_conduits'] = 120   # small slice for a fast Colab demo
cfg['optimization']['pop_size'] = 24
cfg['optimization']['n_gen'] = 15
cfg['optimization']['n_workers'] = 2         # Colab has few cores
yaml.safe_dump(cfg, open('config.yaml', 'w'))
print('demo config set')

!python -m src.build_graph

## 5. Run NSGA-II (small budget)

In [ ]:
!python -m src.run --algo nsga2 --seed 0

## 6. Compare + show the trade-off front

(With one algorithm and one seed the statistics are thin — add `moead`/`ga` and
more seeds to reproduce the full comparison.)

In [ ]:
!python -m src.run --algo moead --seed 0
!python -m src.run --algo ga    --seed 0
!python -m src.compare

from IPython.display import Image, display
for f in ['pareto_fronts.png', 'node_reduction.png']:
    display(Image('outputs/figures/' + f))